# SEA-AD single nuclei Data
This notebook display information about the SEA-AD datasets.
SEA-AD currently contained 2 datasets : single nuclei data from DLPFC and from MTG region.

## Data source
The datasets have been downloaded from [aws](https://sea-ad-single-cell-profiling.s3.amazonaws.com) (see [01-get_seaad_data.R](../scripts/01-get_seaad_data.R))  
The data and metadata was stored in a same .h5ad file (anndata/python based single cell object).  

For each dataset, Cell metadata (containing both donor and cell level information) have been extracted from the objects and save in csv in `outputs/01-SEAAD_data/[BrainRegion]/all_final_RNAseq_nuclei_metadata.csv.gz` (see [01Ai-extract_metadata.py](../scripts/01Ai-extract_metadata.py))


## Cell type annotation
SEA-AD data have already be annotated for cell types and stored in the cell level metadata. 
Here we will first harmonize this annotation to be easily integrated with others dataset. SEA-AD from the Allen institute which have created the [Allen Brain map](https://portal.brain-map.org/) subdivize Excitatory and Inhibitory neurons according the brain layers from which they come from.
We will create a `cell_type` annotation containing these different cell types annotation, but also a `main_cell_type` containing only the main category used in most of the study



In [ ]:
library(data.table)
library(stringr)
library(ggplot2) 
mtd<-fread('../outputs/01-SEAAD_data/DLPFC/all_final_RNAseq_nuclei_metadata.csv.gz')
mtd[,cell_type:=ifelse(str_detect(Class,'Glut'),paste0('Exc_',Subclass),
                       ifelse(str_detect(Class,'GABA'),paste0('Inh_',Subclass),Subclass))]
mtd[,main_cell_type:=str_extract(cell_type,'Oligo|Exc|Inh|Astro|Mic|Endo|VLMC|OPC')]
mtd[,main_cell_type:=factor(main_cell_type,levels = c('Exc','Inh','Oligo','Astro','OPC','Mic','VLMC','Endo'))]
unique(mtd[,.(main_cell_type,Subclass,cell_type)][order(cell_type)])

fwrite(mtd,'../outputs/01-SEAAD_data/DLPFC/all_final_RNAseq_nuclei_metadata.csv.gz')


## Where to find what ?

### Annotated seurat objects for each cell type
Seurat object for each cell type have been created with all metadata and outliers flag in  `outputs/01-SEAAD_data/[BrainRegion]`. Because of the large size of Excitatory and Inhibitory neurons, these cell types have been split by `Subclass`. See [this notebook]('SEAAD_QC.ipynb') for more information.

### Metadata
If you want all the  metadata of each cells, you can find that in `outputs/01-SEAAD_data/[BrainRegion]/all_final_RNAseq_nuclei_metadata.csv.gz` 

For the sample level information only (clinical covariates..), you can find in `outputs/01-SEAAD_data/[BrainRegion]/all_final_RNAseq_nuclei_sample_level_metadata.csv.gz`  

### Pseudobulk data
From this Seurat Object, Pseudobulk data have been generated . For each `Subclass` (in `outputs/01-SEAAD_data/[BrainRegion]`), but also for each main cell type (e.g. Excitatory Neurons Subtypes together), in `outputs/01-SEAAD_data/[BrainRegion]/pseudobulk_main_cell_type`

### small QCed seurat object 
 For each cell type, a small QCed seurat object have been generated, allowing quick exploration/visualization of each cell type. These objects can be found in `outputs/01-SEAAD_data/[BrainRegion]/QC_small`




### Cells Stat

In [7]:
library(data.table)
mtd<-fread('../outputs//01-SEAAD_data/DLPFC/all_final_RNAseq_nuclei_metadata.csv.gz')
mtd[,n.cells.main:=.N,by='main_cell_type']
mtd[,n.cells.subclass:=.N,by='Subclass']
mtd[,n.cells.supertype:=.N,by='Supertype']


unique(mtd[,.(main_cell_type,n.cells.main,cell_type,Subclass,n.cells.subclass,Supertype,n.cells.supertype)][order(-n.cells.main,-n.cells.subclass,-n.cells.supertype)])


main_cell_type,n.cells.main,cell_type,Subclass,n.cells.subclass,Supertype,n.cells.supertype
<chr>,<int>,<chr>,<chr>,<int>,<chr>,<int>
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_1,114872
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_6,59793
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_5,46273
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_13,40232
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_10,32987
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_3,16410
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_12,14317
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_8,12716
Exc,660751,Exc_L2/3 IT,L2/3 IT,341960,L2/3 IT_7,3205


## Data QC and preprocessing
For Data QC (clinical/cellular outliers removal) see [SEAAD_QC.ipynb](SEAAD_QC.ipynb)  
For pseudobulk data creation see [SEAAD_pseudobulk.ipynb](notebooks/SEAAD_pseudobulk.ipynb)


